In [1]:
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorSampler
import json
import math

In [2]:
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import IntegerToBinary

# 1. Define the frequency grid and filter specifications
omegas = np.linspace(0, np.pi, 50) # 50 points from 0 to Nyquist frequency
cutoff = 0.33 * np.pi
stop_start = 0.5 * np.pi

# We will collect the valid points, targets (D), and weights (W)
valid_omegas, targets, weights = [], [], []

for w in omegas:
    if w <= cutoff:
        valid_omegas.append(w) # Ignore w in the transition band
        targets.append(1.0)  # Passband ideal filter is 1
        weights.append(1.0)  # Passband weight
    elif w >= stop_start:
        valid_omegas.append(w) # Ignore w in the transition band
        targets.append(0.0)  # Stopband ideal filter is 0
        weights.append(1.0)  # Stopband weight

# 2. Initialize Matrices for the Objective Function
Q = np.zeros((4, 4)) # Quadratic coefficients
L = np.zeros(4)      # Linear coefficients

# Lower bound = -1.0, Upper bound = 1.0, B = 4 bits (0 to 15)
scale = 2.0 / 15.0   # The (U - L) / (2^B - 1) term for scaling integer variables to the filter coefficients
shift = -1.0         # The Lower Bound (L)

# 3. Calculate the Linear and Quadratic coefficients across the grid
for w, D_k, W_k in zip(valid_omegas, targets, weights):
    # The cosine vector for this specific frequency
    c_k = np.array([1, np.cos(w), np.cos(2*w), np.cos(3*w)])
    
    # Calculate the Constant chunk (K_k)
    K_k = shift * np.sum(c_k) - D_k
    
    # Accumulate Quadratic terms (outer product of c_k with itself)
    Q += W_k * (scale**2) * np.outer(c_k, c_k)
    
    # Accumulate Linear terms
    L += W_k * 2 * K_k * scale * c_k

# 4. Plug it into Qiskit
qp = QuadraticProgram()

# Define the 4 integer variables (0 to 15 represents 4 bits)
for i in range(4):
    qp.integer_var(lowerbound=0, upperbound=15, name=f'x{i}')

# Qiskit natively accepts NumPy matrices for the objective!
qp.minimize(linear=L, quadratic=Q)

# 5. Convert the Integer problem to a Binary QUBO for the Quantum Algorithm
converter = IntegerToBinary()
qubo_QP = converter.convert(qp)

print(f"Problem successfully built!")
print(f"Number of Continuous Integer Variables: {qp.get_num_vars()}")
print(f"Number of Binary Qubits for QAOA: {qubo_QP.get_num_vars()}")

Problem successfully built!
Number of Continuous Integer Variables: 4
Number of Binary Qubits for QAOA: 16


In [3]:


# 1. Initialize classical solver
exact_solver = NumPyMinimumEigensolver()

# 2. Wrap classical solver in opitimization module
exact_optimizer = MinimumEigenOptimizer(exact_solver)

# 3. Solve the QuadraticProgram (your discretized FIR QUBO)
exact_result = exact_optimizer.solve(qubo_QP)

# 5. Extract the important data points
optimal_bits = exact_result.x       # Returns a NumPy array of the optimal bitstring e.g., [1., 1., 0.]
optimal_cost = exact_result.fval    # Returns the objective function value (your minimum error)
opt_status = exact_result.status    # Returns an Enum indicating success/failure

# 6. Store the results in a dictionary for easy saving
saved_results = {
    "optimal_bitstring": [int(bit) for bit in optimal_bits], # Convert floats to ints for clean JSON
    "minimum_error": optimal_cost,
    "status": opt_status.name
}

print("Optimization Results:")
print(json.dumps(saved_results, indent=4))

Optimization Results:
{
    "optimal_bitstring": [
        1,
        1,
        0,
        1,
        0,
        0,
        1,
        1,
        0,
        0,
        0,
        1,
        0,
        1,
        1,
        0
    ],
    "minimum_error": -223.82085485034645,
    "status": "SUCCESS"
}


In [ ]:
# 1. Define the classical optimizer (COBYLA is usually a safe, fast bet)
optimizer = COBYLA(maxiter=100)
sampler = StatevectorSampler()

# 2. Initialize the Quantum Algorithm
# FOR QAOA (EASY): Just specify the reps
qaoa_solver = QAOA(sampler=sampler, optimizer=optimizer, reps=2)

# 3. Wrap it in the Optimization module
qaoa_optimizer = MinimumEigenOptimizer(qaoa_solver) # Just pass QAOA here

# 4. Solve your QuadraticProgram (your discretized FIR QUBO)
result = qaoa_optimizer.solve(qubo_QP)

print(result.prettyprint())

C:\Users\TechG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
C:\Users\TechG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\sparse\linalg\_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
